<a href="https://colab.research.google.com/github/YuuRei00/indonesia-socioeconomic-clustering-2025/blob/main/05_generate_figures_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import warnings
warnings.filterwarnings('ignore')

# ================= KONFIGURASI GLOBAL =================
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight'
})

# Warna konsisten: Resilient=Hijau, Transitional=Biru, Constrained=Merah
COLORS = {'Resilient': '#2E7D32', 'Transitional': '#1565C0', 'Constrained': '#C62828'}

# ================= FIGURE 1: SILHOUETTE SCORES =================
def plot_silhouette():
    k_values = [2, 3, 4, 5, 6, 7, 8]
    silhouette_scores = [0.3754, 0.3257, 0.2989, 0.2607, 0.2526, 0.2220, 0.2182]

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(k_values, silhouette_scores, '-o', color='#034F84', linewidth=2, markersize=8, zorder=3)

    # Annotate semua titik
    for x, y in zip(k_values, silhouette_scores):
        ax.annotate(f"{y:.4f}", (x, y), xytext=(0, 12),
                    textcoords='offset points', ha='center', fontsize=9, fontweight='bold')

    # Highlight k=3 (Selected)
    ax.scatter([3], [0.3257], s=250, facecolors='none', edgecolors=COLORS['Constrained'],
               linewidths=2.5, zorder=5, label='Selected: k=3')
    ax.annotate("Selected: k=3\n(Silhouette = 0.3257)",
                xy=(3, 0.3257), xytext=(4.5, 0.365),
                arrowprops=dict(arrowstyle='->', color=COLORS['Constrained'], lw=1.8),
                color=COLORS['Constrained'], fontsize=10, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor=COLORS['Constrained'], alpha=0.9))

    ax.set_xticks(k_values)
    ax.set_ylim(0.20, 0.40)
    ax.set_yticks(np.arange(0.20, 0.41, 0.04))
    ax.set_xlabel("Number of Clusters (k)", fontweight='bold')
    ax.set_ylabel("Average Silhouette Score", fontweight='bold')
    ax.set_title("Figure 1. Elbow Method Validation: Average Silhouette Score by k", fontweight='bold', pad=15)
    ax.grid(alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(loc='upper right', framealpha=0.9)

    plt.tight_layout()
    plt.savefig("Figure1_Silhouette_Corrected.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Figure 1 saved: Figure1_Silhouette_Corrected.png")

# ================= FIGURE 2: RADAR CHART (6 SUMBU + INVERSI POVERTY) =================
def plot_radar():
    # Data mentah sesuai Tabel 2
    labels = ["UHH\n(Life Expectancy)", "HLS\n(Expected Years)", "RLS\n(Mean Years)",
              "Expenditure\n(Real per Capita)", "GRDP per Capita\n(Current Prices)", "Poverty Rate\n(Inverted *)"]

    # Urutan: Resilient, Transitional, Constrained
    raw_data = np.array([
        [75.17, 14.34, 10.65, 14939, 148790, 6.12],   # Resilient
        [73.46, 13.08, 8.46, 11054, 55820, 10.47],    # Transitional
        [68.24, 11.01, 6.00, 6707, 25266, 29.98]      # Constrained
    ])

    # Min-Max Scaling per kolom
    min_vals = raw_data.min(axis=0)
    max_vals = raw_data.max(axis=0)
    scaled_data = (raw_data - min_vals) / (max_vals - min_vals)

    # Inversi Poverty Rate (kolom ke-6, index 5) agar "lebih tinggi = lebih baik"
    scaled_data[:, 5] = 1 - scaled_data[:, 5]

    # Setup radar
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]  # Tutup polygon

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    cluster_names = ["Resilient (n=136)", "Transitional (n=345)", "Constrained (n=33)"]
    cluster_colors = [COLORS['Resilient'], COLORS['Transitional'], COLORS['Constrained']]

    for i, (vals, name, color) in enumerate(zip(scaled_data, cluster_names, cluster_colors)):
        values = vals.tolist() + vals[:1].tolist()
        ax.plot(angles, values, 'o-', linewidth=2.5, label=name, color=color, markersize=6)
        ax.fill(angles, values, alpha=0.15, color=color)

        # Label nilai untuk Transitional (Cluster tengah) agar tidak terlalu ramai
        if i == 1:
            for angle, value, label in zip(angles[:-1], vals, labels):
                ax.annotate(f"{value:.3f}", xy=(angle, value), xytext=(8, 5),
                           textcoords='offset points', fontsize=8, color=color, fontweight='bold')

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=9, color='grey')
    ax.set_title("Figure 2. Relative Min-Max Scaled Socioeconomic Profile by Cluster\n(Based on Cluster Means)",
                 fontweight='bold', pad=25, fontsize=14)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10, framealpha=0.9)

    # Catatan kaki inversi
    fig.text(0.5, 0.02,
             "* Poverty Rate is inverted (1 − min-max scaled value) so that higher values indicate better socioeconomic conditions.",
             ha='center', fontsize=9, style='italic', color='#555555')
    fig.text(0.5, -0.02,
             "Source: Author's calculation based on the harmonized 2025 dataset.",
             ha='center', fontsize=9, color='#333333')

    plt.tight_layout()
    plt.savefig("Figure2_Radar_Corrected.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Figure 2 saved: Figure2_Radar_Corrected.png")

# ================= FIGURE 3: BAR CHART (MULAI DARI 0) =================
def plot_bar():
    clusters = ["Transitional\n(n=345, 67.1%)", "Resilient\n(n=136, 26.5%)", "Constrained\n(n=33, 6.4%)"]
    counts = [345, 136, 33]
    colors = [COLORS['Transitional'], COLORS['Resilient'], COLORS['Constrained']]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.barh(clusters, counts, color=colors, height=0.6, edgecolor='white', linewidth=1.5)

    # Pastikan sumbu X mulai dari 0
    ax.set_xlim(0, 400)
    ax.set_xticks(np.arange(0, 401, 50))

    # Tambah label di ujung batang
    for bar, count in zip(bars, counts):
        width = bar.get_width()
        ax.text(width + 8, bar.get_y() + bar.get_height()/2,
                f"{count} regions", va='center', ha='left', fontsize=11, fontweight='bold', color='#333333')

    ax.set_xlabel("Number of Regencies/Cities", fontweight='bold', fontsize=12)
    ax.set_title("Figure 3. Distribution of Regencies/Cities Across Socioeconomic Clusters",
                 fontweight='bold', pad=15, fontsize=14)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.savefig("Figure3_Bar_Corrected.png", dpi=300, bbox_inches='tight')
    plt.close()
    print("✅ Figure 3 saved: Figure3_Bar_Corrected.png")

# ================= JALANKAN SEMUA =================
if __name__ == "__main__":
    print("🔄 Generating corrected figures based on Table 2 & 3 data...")
    plot_silhouette()
    plot_radar()
    plot_bar()

🔄 Generating corrected figures based on Table 2 & 3 data...


✅ Figure 1 saved: Figure1_Silhouette_Corrected.png


✅ Figure 2 saved: Figure2_Radar_Corrected.png


✅ Figure 3 saved: Figure3_Bar_Corrected.png
